<a href="https://colab.research.google.com/github/Rogerio-mack/Modelos-de-Linguagem-e-Generativos-2026S1/blob/main/Modelo_Embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo Embedding

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# all-MiniLM-L6-v2: encoder com 6 camadas, ~22M parâmetros
# Produz vetores de 384 dimensões. Muito leve — roda tranquilo em CPU.
# Treinado especificamente para similaridade semântica entre frases.
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Pequeno corpus: cada string é um "documento" que será indexado
# Em um RAG real, seriam parágrafos de PDFs, artigos, etc.
corpus = [
    "A dengue é causada pelo vírus dengue e transmitida pelo Aedes aegypti.",
    "Os sintomas da dengue incluem febre alta, dor de cabeça e dores no corpo.",
    "Não existe vacina amplamente disponível contra a dengue no Brasil.",
    "A malária é transmitida pelo mosquito Anopheles e causada por parasitas.",
    "O tratamento da dengue é sintomático: repouso, hidratação e analgésicos.",
    "Futebol é o esporte mais popular do Brasil.",  # fora do tema — controle
]

# encode(): tokeniza e passa pelo encoder, retornando um vetor por frase
# normalize_embeddings=True: coloca todos os vetores na mesma escala (norma 1)
# Necessário para usar cosine similarity como simples produto escalar
corpus_embeddings = model.encode(corpus, normalize_embeddings=True)

print(f"Shape dos embeddings: {corpus_embeddings.shape}")

Shape dos embeddings: (6, 384)


In [ ]:
corpus_embeddings

array([[-0.04023708,  0.07069238, -0.0160563 , ...,  0.01999174,
        -0.00300812,  0.03083648],
       [-0.0681594 ,  0.04018077, -0.06460534, ..., -0.05061717,
        -0.01676753, -0.03319784],
       [-0.03893494,  0.0336059 , -0.05320413, ..., -0.02162783,
         0.01294312,  0.02238433],
       [-0.03273656,  0.10200678, -0.03623561, ...,  0.04448259,
        -0.01057922,  0.07169934],
       [ 0.03318666,  0.02921379, -0.08846892, ..., -0.04145257,
         0.04078178,  0.03501249],
       [ 0.06261999,  0.02312628, -0.02461625, ...,  0.05711639,
         0.06984777, -0.01194768]], dtype=float32)

In [ ]:
# A pergunta do usuário também vira um vetor no mesmo espaço
query = "O que é dengue?"
query_embedding = model.encode(query, normalize_embeddings=True)

# Produto escalar entre o vetor da query e cada vetor do corpus
# Com vetores normalizados, isso equivale à cosine similarity
# Resultado: um score entre -1 e 1 para cada documento
scores = corpus_embeddings @ query_embedding  # shape: (6,)

# Ordena os documentos do mais para o menos relevante
ranked = sorted(zip(scores, corpus), reverse=True)

print(f"Query: '{query}'\n")
for score, doc in ranked:
    print(f"[{score:.3f}] {doc}")

Query: 'O que é dengue?'

[0.760] A dengue é causada pelo vírus dengue e transmitida pelo Aedes aegypti.
[0.732] Não existe vacina amplamente disponível contra a dengue no Brasil.
[0.729] Os sintomas da dengue incluem febre alta, dor de cabeça e dores no corpo.
[0.718] O tratamento da dengue é sintomático: repouso, hidratação e analgésicos.
[0.536] A malária é transmitida pelo mosquito Anopheles e causada por parasitas.
[0.475] Futebol é o esporte mais popular do Brasil.


In [ ]:
query = "Quais são os sintomas da dengue?"
query_embedding = model.encode(query, normalize_embeddings=True)

scores = corpus_embeddings @ query_embedding  # shape: (6,)

ranked = sorted(zip(scores, corpus), reverse=True)

print(f"Query: '{query}'\n")
for score, doc in ranked:
    print(f"[{score:.3f}] {doc}")

Query: 'Quais são os sintomas da dengue?'

[0.775] Os sintomas da dengue incluem febre alta, dor de cabeça e dores no corpo.
[0.727] O tratamento da dengue é sintomático: repouso, hidratação e analgésicos.
[0.695] Não existe vacina amplamente disponível contra a dengue no Brasil.
[0.636] A dengue é causada pelo vírus dengue e transmitida pelo Aedes aegypti.
[0.468] Futebol é o esporte mais popular do Brasil.
[0.453] A malária é transmitida pelo mosquito Anopheles e causada por parasitas.


In [ ]:
query = "O que é dengue?"
query_embedding = model.encode(query, normalize_embeddings=True)
scores = corpus_embeddings @ query_embedding  # shape: (6,)
ranked = sorted(zip(scores, corpus), reverse=True)

print(f"Query: '{query}'\n")
for score, doc in ranked:
    print(f"[{score:.3f}] {doc}")

Query: 'O que é dengue?'

[0.760] A dengue é causada pelo vírus dengue e transmitida pelo Aedes aegypti.
[0.732] Não existe vacina amplamente disponível contra a dengue no Brasil.
[0.729] Os sintomas da dengue incluem febre alta, dor de cabeça e dores no corpo.
[0.718] O tratamento da dengue é sintomático: repouso, hidratação e analgésicos.
[0.536] A malária é transmitida pelo mosquito Anopheles e causada por parasitas.
[0.475] Futebol é o esporte mais popular do Brasil.


In [ ]:
# Em um pipeline RAG, passaríamos só os top-k documentos para o modelo causal
# Isso evita encher o contexto com informação irrelevante
TOP_K = 3
top_docs = [doc for _, doc in ranked[:TOP_K]]

print("Contexto que seria enviado ao modelo gerador:\n")
for i, doc in enumerate(top_docs, 1):
    print(f"{i}. {doc}")

Contexto que seria enviado ao modelo gerador:

1. A dengue é causada pelo vírus dengue e transmitida pelo Aedes aegypti.
2. Não existe vacina amplamente disponível contra a dengue no Brasil.
3. Os sintomas da dengue incluem febre alta, dor de cabeça e dores no corpo.
